# 19장 · RAG 체인 · RunnableLambda와 assign으로 검색 결과 전달하기

RunnablePassthrough.assign은 입력 딕셔너리의 기존 값을 유지하면서 새 키에 계산 결과를 붙입니다. RAG에서는 질문을 보존하면서 검색한 문단을 추가할 때 씁니다.

## assign으로 새 키 하나 추가하기

`assign(length=...)`는 입력 딕셔너리를 받아 length 값을 계산해 붙이는 단계를 만듭니다. 아래에서 `row`로 들어오는 값은 `{"word": "사과"}`입니다. `invoke` 뒤에 원래 키와 새로운 키가 함께 남는지 확인하세요. 처음 쓰는 커널에는 `%pip install langchain-core`가 필요합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

with_length = RunnablePassthrough.assign(length=lambda row: len(row["word"]))
print(with_length.invoke({"word": "사과"}))

{'word': '사과', 'length': 2}


`word`가 그대로 남고 `length`에 2가 들어갑니다. 다음 단계는 같은 사전에서 단어와 길이를 모두 읽을 수 있습니다.

## 응용 실습 · 검색 문단을 답변에 전달하기

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))
from portfolio import load_step, save_step, page_id


### 저장한 검색 결과를 LCEL 답변 체인에 넣기

18장에서 D04를 검색해 보관한 원문을 사용합니다. 검색 요청을 다시 보내지 않고 같은 문단으로 답을 생성합니다. 14장의 질문·자료 양식이 여기서 다시 쓰입니다.

> **D04**　2027년 4월 15일 기준으로 A형의 파일이 나뉘는 간격은 얼마이고 그때 녹음도 멈추나요?

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rag_lab import chat_model
run = load_step(root, "reopened")
source_template = load_step(root, "prompt")["template"]
question_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 쪽에서 확인한 내용에만 답하고 인용한 [ID]를 그대로 붙이세요. 없으면 확인 보류라고 답하세요."),
    ("human", source_template),
])
evidence = "\n\n".join(f"[{row['id']}] {row['text']}" for row in run["documents"])
answer_chain = question_prompt | chat_model() | StrOutputParser()
reply = answer_chain.invoke({"question": run["question"], "context": evidence})
save_step(root, "baseline-answer", {**run, "answer": reply})
print(reply)

`invoke`의 두 입력은 원래 질문과 18장의 실제 후보입니다. 답에 적힌 ID가 어느 후보인지 찾고 본문에서 분할 간격을 읽어 보세요. 20장에서 날짜 필터를 넣기 전까지는 구판도 모델에 들어갈 수 있습니다.

### 실습 · 실제로 전달한 쪽을 열어 보기

응답에 포함된 후보 ID를 찾아 파일명과 쪽수를 출력하세요. 모델이 인용을 빼먹으면 빈 목록이 나와야 합니다.

In [ ]:
# run["documents"]에서 [ID]가 reply에 있는 후보만 고르세요.

<details><summary>정답 코드·출력·풀이</summary><pre><code>cited = [r for r in run[&quot;documents&quot;] if f&quot;[{r[&#x27;id&#x27;]}]&quot; in reply]
print([(r[&quot;source&quot;], r[&quot;page&quot;]) for r in cited])</code></pre><p>제공 자료로 확인할 값</p><pre>[(&#x27;recorder-a-1.1.pdf&#x27;, 2)]</pre><p>기본 D04의 정답 쪽을 인용한 예입니다. 실제 출력이 다르면 인용과 원문을 함께 남기세요.</p></details>

## 검색 문단을 context 문자열로 만들기

질문은 검색 함수와 프롬프트에 각각 한 번씩 들어갑니다. `retrieve(index, question, k=3)`로 문단을 가져온 뒤, 아래의 `context = ...` 줄은 각 문단의 ID와 본문을 한 문자열로 묶습니다. `prompt.invoke(payload)`로 실제로 들어갈 질문과 자료를 먼저 읽을 수 있습니다.

In [2]:
from rag_lab import open_index, retrieve
index = open_index()
question = "2027년 5월 B형은 표시등을 숨긴 뒤 어디서 녹음 상태를 확인하나요?"
documents = retrieve(index, question, k=3)
for d in documents:
    print(d["id"], d["text"][:80])

recorder-b-1.0.pdf#p1 들소리 B형   /   휴대용 녹음기
녹음 표시등
1.0판    적용 기간   2027년 4월 1일부터
표시등 설정
녹음 전에 설정에서 표시등을
recorder-a-1.0.pdf#p1 들소리 A형   /   휴대용 녹음기
녹음 표시등
1.0판    적용 기간   2027년 3월 1일 ~ 3월 31일
표시등 설정
녹음 중에는 앞
recorder-a-1.1.pdf#p1 들소리 A형   /   휴대용 녹음기
녹음 표시등
1.1판    적용 기간   2027년 4월 1일부터
표시등 설정
녹음 중에는 앞면의 빨간 표


### join으로 ID와 본문을 함께 전달하기

프롬프트의 자료 칸에는 문자열 하나를 넣습니다. `"\n\n".join(...)`은 여러 문자열 사이에 빈 줄을 두고 합칩니다.

In [ ]:
parts = ["[p1] 빨강", "[p2] 파랑"]
print("\n\n".join(parts))

[p1] 빨강

[p2] 파랑


두 문단 사이에 빈 줄이 생기고 각 문단 앞에 ID가 남습니다. 다음 셀에서는 이 두 요소를 검색 결과의 `id`와 `text`에서 읽습니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rag_lab import chat_model

context = "\n\n".join(f"[{d['id']}]\n{d['text']}" for d in documents)
prompt = ChatPromptTemplate.from_messages([
    ("system","자료 안에서 기종과 시점이 맞는 사실만 쓰고, 근거 ID를 대괄호로 인용하세요. "
     "확인할 수 없으면 무엇이 부족한지 밝혀 주세요."),
    ("human","원래 질문: {question}\n자료: {context}")])
payload = {"question": question, "context": context}
print(prompt.invoke(payload).to_messages()[-1].content)

원래 질문: 2027년 5월 B형은 표시등을 숨긴 뒤 어디서 녹음 상태를 확인하나요?
자료: [recorder-b-1.0.pdf#p1]
들소리 B형   /   휴대용 녹음기
녹음 표시등
1.0판    적용 기간   2027년 4월 1일부터
표시등 설정
녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸세요. 녹음을
시작해도 앞면에 빨간불이 들어오지 않습니다.
화면과 표시등
이미 녹음 중이면 표시등 설정을 바꿀 수 없습니다. 녹음을 끝낸
뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작하세요.
녹음 상태 확인
표시등을 숨겼을 때는 화면의 녹음 시간을 확인하세요. 시간이
늘어나면 녹음 중입니다.
가상 제품 설명서   B형 1.0판
1 / 2쪽

[recorder-a-1.0.pdf#p1]
들소리 A형   /   휴대용 녹음기
녹음 표시등
1.0판    적용 기간   2027년 3월 1일 ~ 3월 31일
표시등 설정
녹음 중에는 앞면의 빨간 표시등이 켜집니다. 녹음하는 동안
표시등을 끄는 설정은 없습니다.
화면과 표시등
화면 밝기를 낮춰도 표시등은 켜져 있습니다. 조용한 방에서
화면을 어둡게 하고 싶을 때는 설정에서 화면 밝기를 낮추세요.
녹음 상태 확인
종료 버튼을 누르면 녹음이 끝나고 표시등이 꺼집니다. 이때
마지막 파일을 저장합니다.
가상 제품 설명서   A형 1.0판
1 / 2쪽

[recorder-a-1.1.pdf#p1]
들소리 A형   /   휴대용 녹음기
녹음 표시등
1.1판    적용 기간   2027년 4월 1일부터
표시등 설정
녹음 중에는 앞면의 빨간 표시등이 켜집니다. 녹음하는 동안
표시등을 끄는 설정은 없습니다.
화면과 표시등
화면 밝기를 낮춰도 표시등은 켜져 있습니다. 조용한 방에서
화면을 어둡게 하고 싶을 때는 설정에서 화면 밝기를 낮추세요.
녹음 상태 확인
종료 버튼을 누르면 녹음이 끝나고 표시등이 꺼집니다. 이때
마지막 파일을 저장합니다.
가상 제품 설명서   A형 1.1판
1 / 2쪽


출력에서 B형의 ‘화면의 녹음 시간’ 문장을 직접 찾으세요. 이 문장이 없다면 검색 실패를 먼저 조사합니다. 문장이 있는데 모델 답이 “앞면 빨간불을 보세요”라고 하면, 어긋난 대상 또는 답변 지시를 살펴봅니다.

In [4]:
chain = prompt | chat_model() | StrOutputParser()
print(chain.invoke(payload))

2027년 5월 B형(1.0판)은 녹음 표시등을 숨긴 뒤, 녹음 상태를 화면의 녹음 시간을 확인하여 알 수 있습니다. 녹음 시간이 늘어나면 녹음 중임을 의미합니다. 표시등을 숨기면 앞면에 빨간불이 들어오지 않으므로, 화면의 녹음 시간을 통해 상태를 확인해야 합니다[recorder-b-1.0.pdf#p1].


## RunnablePassthrough.assign으로 문단과 답 추가하기

실제 화면에서는 검색 문단을 다시 호출해 가져오지 않도록 질문과 문단, 답변을 하나로 돌려줍니다. `assign`은 이미 있던 키를 남긴 채 새 키를 추가합니다. 아래 첫 단계가 documents를 넣어야 다음 단계가 그것을 읽을 수 있습니다.

RunnableLambda는 평범한 Python 함수를 invoke할 수 있는 한 단계로 감쌉니다. `lambda item: ...`에서 item은 앞 단계에서 받은 딕셔너리입니다. 첫 assign을 지나면 question에 documents가 추가되고, 둘째 assign에서는 두 칸을 모두 읽어 answer를 붙입니다. 원래 질문 문자열을 검색 결과로 덮어쓰지 않으므로 화면이 어떤 질문에 대한 결과인지 표시할 수 있습니다.

`RunnableLambda`는 파이썬 함수를 LCEL 단계로 감쌉니다. `assign(documents=...)`은 입력 딕셔너리의 기존 키를 유지하면서 계산한 값을 `documents`에 담습니다. 다음 `assign(answer=...)`에서 검색 결과와 원래 질문을 함께 꺼낼 수 있는 까닭입니다. 이 순서를 바꾸면 답변 단계가 읽을 `documents`가 아직 없습니다.

In [5]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from rag_lab import answer_from_documents

with_sources = RunnablePassthrough.assign(documents=RunnableLambda(
    lambda item: retrieve(index, item["question"], k=3)))
rag = with_sources | RunnablePassthrough.assign(answer=RunnableLambda(
    lambda item: answer_from_documents(item["question"], item["documents"])))
result = rag.invoke({"question": question})
print(result["answer"])
print("실제 후보", [d["id"] for d in result["documents"]])

2027년 5월 B형은 표시등을 숨긴 뒤 화면의 녹음 시간을 확인하여 녹음 상태를 확인합니다. 시간이 늘어나면 녹음 중임을 알 수 있습니다.[recorder-b-1.0.pdf#p1]
실제 후보 ['recorder-b-1.0.pdf#p1', 'recorder-a-1.0.pdf#p1', 'recorder-a-1.1.pdf#p1']


## 검색 결과를 통째로 바꾸는 작은 진단

아까의 질문은 그대로 둔 채 B형 2쪽만 주어 봅니다. 프로그램이 PDF를 열었다는 점은 같지만, 파일 저장과 복사 안내만으로는 표시등을 숨긴 뒤 어디서 상태를 확인하는지 답할 수 없습니다. 여기서는 **질문을 바꾸지 않고 문서 한 개만 교체**해 근거가 달라지는 원인을 분리합니다.

In [6]:
from rag_lab import SOURCES, read_pages
wrong = read_pages(SOURCES / "recorder-b-1.0.pdf")[1]
print("전달한 쪽", wrong["id"])
print(answer_from_documents(question, [wrong]))

전달한 쪽 recorder-b-1.0.pdf#p2
확인 보류: 2027년 5월 B형에서 표시등을 숨긴 뒤 녹음 상태를 확인하는 방법에 대한 정보가 자료에 없습니다.[recorder-b-1.0.pdf#p2]


출처가 실제로 존재하는데도 해당 사실이 없을 수 있습니다. 모델이 답을 만들었다면 그 문장을 원문에서 찾아 보세요. 없으면 실패이며, 출처 이름을 정확히 썼다는 이유만으로 맞게 처리하지 않습니다. 답을 보류한 경우에는 어느 정보가 더 필요한지 학생이 설명할 수 있어야 합니다.

## 실습 · 질문과 인용의 연결 끊긴 곳 찾기

같은 B형 1쪽으로 “숨김을 언제 설정하는지”와 “숨긴 뒤 녹음 상태를 어디서 보는지”를 각각 묻습니다. 두 답이 같은 페이지를 인용하더라도 각각 다른 문장이 필요합니다. 모델이 인용한 쪽을 열고 답을 지지하는 문장을 옮기세요.

In [7]:
questions_to_try = ["2027년 5월 B형에서 표시등 숨김은 언제 설정하나요?",
                    "2027년 5월 B형에서 표시등을 숨긴 뒤 녹음 중인지 어떻게 확인하나요?"]
# 질문마다 rag.invoke에 question 키로 넘기세요. 결과에서 answer와 근거 쪽을 꺼내 읽으세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>for q in [&quot;2027년 5월 B형에서 표시등 숨김은 언제 설정하나요?&quot;,
          &quot;2027년 5월 B형에서 표시등을 숨긴 뒤 녹음 중인지 어떻게 확인하나요?&quot;]:
    run = rag.invoke({&quot;question&quot;: q})
    print(q, run[&quot;answer&quot;])
    print([p[&quot;id&quot;] for p in run[&quot;documents&quot;]])</code></pre><p>첫 답은 녹음 시작 전 설정을, 두 번째 답은 화면의 녹음 시간 증가를 다룹니다. 인용 페이지가 맞아도 질문과 관계없는 문장만 답했다면 불충분합니다.</p><p>집필 때 저장한 두 질문의 답변 본문입니다. 위 셀을 실행하면 각 답 아래에 이번 검색의 후보 ID도 나옵니다.</p><pre>2027년 5월 B형에서 표시등 숨김은 언제 설정하나요? 2027년 5월 B형에서 표시등 숨김은 녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸는 시점에 설정합니다. 이미 녹음 중이면 표시등 설정을 바꿀 수 없고, 녹음을 끝낸 뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작해야 합니다.[recorder-b-1.0.pdf#p1]
2027년 5월 B형에서 표시등을 숨긴 뒤 녹음 중인지 어떻게 확인하나요? 2027년 5월 B형에서 표시등을 숨긴 뒤 녹음 중인지 확인하려면, 화면의 녹음 시간을 확인하세요. 시간이 늘어나면 녹음 중입니다.[recorder-b-1.0.pdf#p1]</pre></details>

## 독립 실습 · 답을 얻은 뒤 인용을 다시 열기

B형의 표시등을 숨긴 채로 녹음 중인지 확인하는 질문에 답을 받은 뒤, 응답에 인용된 쪽과 실제로 전달된 후보를 대조합니다. 다음에는 B형 2쪽만 모델에 주고 같은 질문을 보냅니다. 실패가 생기면 모델이 받은 문장 중 무엇이 빠졌는지 표시하세요.

| 질문 | 실제 전달한 ID | 생성한 답 | 인용 ID | 원문에서 표시한 문장 |
|---|---|---|---|---|
| 첫 실행 | | | | |
| B형 2쪽만 전달 | | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>B형 1.0판 1쪽에는 표시등을 숨긴 경우 화면의 녹음 시간이 늘어나는지 확인하라고 적혀 있습니다. B형 2쪽만 보낸 경우에는 파일 저장과 복사 문장뿐입니다. 2쪽의 ID가 실제 후보라는 사정만으로 그 쪽을 녹음 상태 확인의 근거로 쓸 수 없습니다.</p></details>

## 다음 수업에 가져갈 것

20장은 `my-course/rag/baseline-answer.json`의 실제 후보를 다시 읽습니다. D04 답에 인용한 쪽이 2027년 4월의 A형 판본인지 직접 확인하세요.

이 체인을 이후 공용 함수 answer_question으로 다시 사용합니다. 첫 실행의 question·documents·answer를 남기세요. 다음 장은 이름이 같은 문서의 두 판본에서 어느 쪽을 써야 하는지 다룹니다.